# 속성별 리뷰 문장 분리 모델
긴 리뷰에서 속성별 표현을 꺼내는 것.

```text
입력: 촉촉하지만 지속력이 짧고 모공에 끼어요.

보습력/수분감 → 촉촉하지만
지속력/유지력 → 지속력이 짧고
윤기/피부(톤) → 모공에 끼어요
```
데이터 확인 → 설정 확인 → 학습 → 테스트 순서

## 1. 준비 파일

- `data/train.jsonl`: 학습 데이터
- `data/validation.jsonl`: 학습 중 확인용 데이터
- `data/test.jsonl`: 최종 확인용 데이터
- `data/labels.json`: 기존 모델에서 쓰던 26개 속성
- `config.json`: 모델과 학습 경로 설정
- `scripts/train.py`: 실제 학습 코드
- `predict.py`: 새 리뷰 테스트 코드

In [1]:
from pathlib import Path
import json
import subprocess
import sys

# notebooks 폴더에서 열었을 때와 프로젝트 루트에서 열었을 때 모두 대응
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("프로젝트 위치:", ROOT)

프로젝트 위치: /home/bteam/aspect_sentence_split


## 2. GPU 확인

`True`와 GPU 이름이 나오면 준비 완료. `False`면 현재 노트북이 GPU 환경에 연결되지 않은 상태.

In [2]:
import torch

print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

CUDA 사용 가능: True
GPU 이름: NVIDIA GeForce RTX 3060


## 3. 데이터 한 건 확인

`text`가 리뷰 원문 `spans` 원문에서 잘라낸 속성별 표현.

`start`, `end`는 원문 안에서 표현이 있는 위치. 학습 코드가 이 위치를 보고 정답을 만듦.

In [3]:
train_path = ROOT / "data" / "train.jsonl"

with train_path.open(encoding="utf-8") as file:
    sample = json.loads(next(file))

print("리뷰 ID:", sample["review_id"])
print("카테고리:", sample["category"])
print("\n리뷰 원문:\n", sample["text"])
print("\n속성별 표현:")
for span in sample["spans"]:
    print("-", span["aspect"], "→", span["text"])

리뷰 ID: 1023184
카테고리: 스킨케어

리뷰 원문:
 클렌징 오일과 클렌징 폼으로 두 번 하는 세안, 사실 번거롭잖아요? 두 가지를 한 번에 할 수 있는 클렌징 제품이 있다고 해서 사용해 보고 후기 올립니다. 제가 구매한 제품은 대용량이라 일단 용량이 넉넉해서 좋아요. 가격도 아주 착한데, 감사하게도 샘플까지 많이 챙겨 주셨어요. 이러면 사용 전부터 마음이 훈훈해지죠?ㅎㅎ 사용해 보니 예민한 피부에 자극 없이 순하고 세정력도 좋은 것 같아요. 진한 메이크업까지 한 번에 지워지지는 정도는 아니지만, 데일리 메이크업 정도는 부드럽게 잘 닦이는 느낌이랍니다. 용기는 펌핑 방식이라 사용하기가 좋고, 구성품으로 실리콘브러시가 들어 있어서 블랙헤드 문지르기에 아주 편해요. 이만하면 팔방미인이라 해야겠어요. 클렌징 제품은 어쩐지 매번 새로운 걸 써 보게 돼요. 궁금하니까요! 이 제품도 다음에 다시 만날지 어떨지 아직은 모르지만~ 만족하며 사용하고 있답니다.

속성별 표현:
- 편의성/활용성 → 두 가지를 한 번에 할 수 있는
- 용량/개수 → 대용량이라 일단 용량이 넉넉해서 좋아요.
- 가격 → 가격도 아주 착한데,
- 제품구성 → 감사하게도 샘플까지 많이 챙겨 주셨어요.
- 자극성 → 예민한 피부에 자극 없이 순하고
- 기능/효과 → 세정력도 좋은 것 같아요.
- 기능/효과 → 진한 메이크업까지 한 번에 지워지지는 정도는 아니지만,
- 기능/효과 → 데일리 메이크업 정도는 부드럽게 잘 닦이는 느낌이랍니다.
- 용기 → 용기는 펌핑 방식이라 사용하기가 좋고,
- 제품구성 → 구성품으로 실리콘브러시가 들어 있어서 블랙헤드 문지르기에 아주 편해요.


## 4. 데이터 크기 확인

현재 준비된 데이터 규모:

- 학습 리뷰 약 3만 6천 건
- 학습 속성 표현 약 10만 건
- 속성 26개

아래 셀은 실제 파일 기준으로 개수를 다시 확인하는 용도.

In [4]:
def count_jsonl(path):
    reviews = 0
    spans = 0
    with path.open(encoding="utf-8") as file:
        for line in file:
            row = json.loads(line)
            reviews += 1
            spans += len(row["spans"])
    return reviews, spans

for split in ["train", "validation", "test"]:
    reviews, spans = count_jsonl(ROOT / "data" / f"{split}.jsonl")
    print(f"{split:10s} 리뷰 {reviews:,}개 / 속성 표현 {spans:,}개")

train      리뷰 36,322개 / 속성 표현 101,780개
validation 리뷰 2,440개 / 속성 표현 8,545개
test       리뷰 2,441개 / 속성 표현 8,718개


## 5. 학습 설정 확인

처음에는 아래 값 그대로 사용하면 됨.

- `base_model`: 한국어를 학습한 기본 모델
- `max_length`: 한 번에 읽을 최대 길이
- `negative_aspects_per_review`: 리뷰에 없는 속성도 없다고 학습시키는 개수

처음 학습에서는 설정을 많이 바꾸지 않는 쪽이 결과 비교에 편함.

In [5]:
config_path = ROOT / "config.json"
config = json.loads(config_path.read_text(encoding="utf-8"))
print(json.dumps(config, ensure_ascii=False, indent=2))

{
  "source_data_dir": "../aspect_sentiment/data/aspect",
  "prepared_data_dir": "data",
  "model_output_dir": "models/aspect_span_extractor",
  "base_model": "klue/roberta-base",
  "max_length": 512,
  "negative_aspects_per_review": 2,
  "seed": 42
}


## 6. 모델 구조를 아주 짧게 이해하기

모델에 리뷰만 넣는 게 아니라 `속성명 + 리뷰`를 같이 넣음.

```text
속성명: 지속력/유지력
리뷰: 촉촉하지만 지속력이 짧고 모공에 끼어요.
정답: 지속력이 짧고
```

같은 리뷰에 `보습력/수분감`을 넣으면 `촉촉하지만`을 찾도록 학습.

이 방식을 쓰는 이유는 하나의 표현이 두 속성에 겹쳐도 둘 다 찾을 수 있기 때문. 모델 파일은 하나만 만들어짐.

## 7. 학습 시작

아래 셀을 실행하면 `scripts/train.py`가 실행됨. 복잡한 토큰 처리와 학습 코드는 스크립트 안에 정리해 둔 상태.

학습 중에는 `eval_span_f1`을 확인하면 됨. 값이 높을수록 속성 구절을 더 잘 찾는다는 뜻.

GPU 메모리가 부족하다는 오류가 나오면 `scripts/train.py`의 `per_device_train_batch_size=8`을 `4` 또는 `2`로 낮추면 됨.


처음 실행할 때 보일 수 있는 메시지:

- `HF_TOKEN` 경고: 로그인 없이 모델을 내려받았다는 안내. 오류가 아님
- `UNEXPECTED lm_head`: 기존 언어모델용 출력층을 사용하지 않는다는 안내. 오류가 아님
- `MISSING classifier`: 새 분류층을 처음부터 학습한다는 안내. 정상 동작

진행률이 `0%`에서 숫자가 올라가기 시작하면 학습이 정상적으로 시작된 상태.

In [10]:
# 실제 학습이 시작되는 셀
# 오래 걸릴 수 있으므로 GPU 서버에서 실행
subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "train.py")],
    cwd=ROOT,
    check=True,
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8018.53it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: klue/roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
  0%|          | 51/29409 [00:10<1:37:15,  5.03it/s]

{'loss': '1.379', 'grad_norm': '9.141', 'learning_rate': '1.997e-05', 'epoch': '0.005101'}


  0%|          | 100/29409 [00:20<1:36:29,  5.06it/s]

{'loss': '0.8519', 'grad_norm': '6.244', 'learning_rate': '1.993e-05', 'epoch': '0.0102'}


  1%|          | 151/29409 [00:30<1:37:12,  5.02it/s]

...

{'eval_loss': '0.08035', 'eval_span_precision': '0.8774', 'eval_span_recall': '0.8633', 'eval_span_f1': '0.8703', 'eval_runtime': '40.13', 'eval_samples_per_second': '293', 'eval_steps_per_second': '73.26', 'epoch': '3'}



100%|██████████| 29409/29409 [1:39:42<00:00,  5.11it/s]

{'train_runtime': '5983', 'train_samples_per_second': '78.65', 'train_steps_per_second': '4.916', 'train_loss': '0.2545', 'epoch': '3'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.81it/s]


CompletedProcess(args=['/home/bteam/aspect_sentence_split/.venv/bin/python', '/home/bteam/aspect_sentence_split/scripts/train.py'], returncode=0)

## 8. 학습 결과 위치

학습이 끝나면 아래 폴더가 생김.

```text
models/aspect_span_extractor/
```

모델, 토크나이저, 속성 목록이 같이 저장됨. 서비스 서버로 옮길 때 이 폴더 전체가 필요.

In [11]:
model_dir = ROOT / "models" / "aspect_span_extractor"

if model_dir.exists():
    print("학습 모델 확인 완료:", model_dir)
    for path in sorted(model_dir.iterdir()):
        print("-", path.name)
else:
    print("아직 학습 모델이 없음. 7번 학습 셀을 먼저 실행하면 됨.")

학습 모델 확인 완료: /home/bteam/aspect_sentence_split/models/aspect_span_extractor
- aspects.json
- checkpoint-19606
- checkpoint-29409
- checkpoint-9803
- config.json
- model.safetensors
- tokenizer.json
- tokenizer_config.json
- training_args.bin


## 9. 새 리뷰 테스트

학습이 끝난 뒤 실행하는 셀. 결과에는 학습용 속성명과 원문에서 찾은 표현이 나옴.

In [21]:
review_text = "첫 발색은 맑고 촉촉하게 올라오지만 여러 번 덧바를수록 더욱 선명하고 고급스러운 컬러감이 표현되어 다양한 립 메이크업을 즐길 수 있었어요. 글레이즈 특유의 탱글한 광택이 입술을 더욱 볼륨감 있어 보이게 해주며, 각질 부각 없이 매끈하게 밀착되는 점도 만족스러웠습니다. 시간이 지나도 자연스러운 착색이 남아 컬러 유지력이 좋았고, 건조함이 적어 편안하게 사용할 수 있었습니다. 데일리 메이크업부터 분위기 있는 메이크업까지 모두 잘 어울리는 컬러로, 촉촉한 광택 립을 좋아하는 분들께 추천하고 싶은 틴트입니다."

completed = subprocess.run(
    [
        sys.executable,
        str(ROOT / "predict.py"),
        "--model-dir",
        str(model_dir),
        "--text",
        review_text,
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print(completed.stdout)
if completed.stderr:
    print(completed.stderr)

[
  {
    "aspect": "밀착력/접착력",
    "aspect_phrase": "매끈하게 밀착되는 점도 만족스러웠습니다."
  },
  {
    "aspect": "발색력",
    "aspect_phrase": "첫 발색은 맑고 촉촉하게 올라오지만"
  },
  {
    "aspect": "발색력",
    "aspect_phrase": "여러 번 덧바를수록 더욱 선명하고 고급스러운 컬러감이 표현되어"
  },
  {
    "aspect": "발색력",
    "aspect_phrase": "자연스러운 착색이 남아"
  },
  {
    "aspect": "보습력/수분감",
    "aspect_phrase": "각질 부각 없이"
  },
  {
    "aspect": "보습력/수분감",
    "aspect_phrase": "건조함이 적어 편안하게 사용할 수 있었습니다."
  },
  {
    "aspect": "색상",
    "aspect_phrase": "데일리 메이크업부터 분위기 있는 메이크업까지 모두 잘 어울리는 컬러로,"
  },
  {
    "aspect": "윤기/피부(톤)",
    "aspect_phrase": "촉촉한 광택 립을 좋아하는 분들께 추천하고 싶은 틴트입니다."
  },
  {
    "aspect": "지속력/유지력",
    "aspect_phrase": "시간이 지나도 자연스러운 착색이 남아"
  },
  {
    "aspect": "지속력/유지력",
    "aspect_phrase": "컬러 유지력이 좋았고,"
  },
  {
    "aspect": "탄력",
    "aspect_phrase": "입술을 더욱 볼륨감 있어 보이게 해주며,"
  },
  {
    "aspect": "편의성/활용성",
    "aspect_phrase": "다양한 립 메이크업을 즐길 수 있었어요."
  }
]


Loading weights: 100%|██████████| 199/199 [00:00<00:

## 10. 서비스에서 속성 이름 바꾸기

모델의 속성명은 그대로 두고 화면이나 DB에 저장하기 직전에 이름만 변경.

```python
SERVICE_LABEL_MAP = {
    "보습력/수분감": "수분감",
    "지속력/유지력": "지속력",
    "밀착력/접착력": "밀착력",
    "윤기/피부(톤)": "피부표현",
}
```

서비스 이름이 바뀌어도 모델을 다시 학습할 필요 없음.